# Setup Trading Environment

In [68]:

# Access API keys
api_key = None
secret_key = None

#### We use paper environment for this example ####
paper = True # Please do not modify this. This example is for paper trading only.
####

# Below are the variables for development this documents
# Please do not change these variables
trade_api_url = None
trade_api_wss = None
data_api_url = None
stream_data_wss = None

In [69]:
import os
from dotenv import load_dotenv

if api_key is None:
    api_key = os.environ.get('API_KEY')

if secret_key is None:
    secret_key = os.environ.get('SECRET_KEY')

if trade_api_url is None:
    trade_api_url = os.environ.get('BASE_URL')





In [70]:
import json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from alpaca.trading.client import TradingClient
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.historical.crypto import CryptoHistoricalDataClient
from alpaca.trading.stream import TradingStream
from alpaca.data.live.crypto import CryptoDataStream

from alpaca.data.requests import (
    CryptoBarsRequest,
    CryptoQuoteRequest,
    CryptoTradesRequest,
    CryptoLatestQuoteRequest
    )
from alpaca.trading.requests import (
    GetAssetsRequest,
    MarketOrderRequest,
    LimitOrderRequest,
    StopLimitOrderRequest,
    GetOrdersRequest,
    ClosePositionRequest
)
from alpaca.trading.enums import (
    AssetClass,
    AssetStatus,
    OrderSide,
    OrderType,
    TimeInForce,
    QueryOrderStatus
)
from alpaca.common.exceptions import APIError

### Initialize Alpaca API Trading Clients

In [71]:
# setup clients
trade_client = TradingClient(api_key=api_key, 
                             secret_key=secret_key,
                             paper=paper, 
                             url_override=trade_api_url)

# Fetch and Clean Historical Data

In [72]:
from alpaca.data.historical import CryptoHistoricalDataClient
from alpaca.data.requests import CryptoBarsRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime, timedelta
import pandas as pd
import numpy as np


# no keys required for crypto data
client = CryptoHistoricalDataClient()


# Function to fetch historical data
def get_historical_data(symbol, years_back=5, timeframe=TimeFrame.Day):
    """
    Fetch historical crypto data for a given symbol and timeframe.

    Parameters:
    - symbol (str): The crypto symbol to fetch (e.g., "BTC/USD").
    - years_back (int): Number of years of data to fetch.
    - timeframe (TimeFrame): The timeframe for bars (default: TimeFrame.Day).

    Returns:
    - pd.DataFrame: Crypto historical data.
    """
    start_date = datetime.today() - timedelta(days=365 * years_back)
    end_date = datetime.today()

    request_params = CryptoBarsRequest(
        symbol_or_symbols=[symbol],
        timeframe=timeframe,
        start=start_date,
        end=end_date
    )

    bars = client.get_crypto_bars(request_params)
    
    # Convert to DataFrame
    df = bars.df

    # Format the timestamp correctly
    df.index = pd.MultiIndex.from_arrays([
        df.index.get_level_values('symbol'),
        pd.to_datetime(df.index.get_level_values('timestamp')).tz_localize(None)
    ], names=['symbol', 'timestamp'])

    return df


# Fetch data separately
btc_d = get_historical_data("BTC/USD")
eth_d = get_historical_data("ETH/USD")

btc_data = btc_d.copy()
eth_data = eth_d.copy()
btc_data = btc_data.reset_index()
eth_data = eth_data.reset_index()

pd.set_option('display.width', 200)  # Set a large width
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)  # Prevent line wrapping

# Display first few rows
print(btc_data.head())
print(eth_data.head())


    symbol           timestamp      open      high       low     close       volume  trade_count          vwap
0  BTC/USD 2021-01-01 06:00:00  29255.71  29682.29  28707.56  29676.79   848.874030      29639.0  29316.444625
1  BTC/USD 2021-01-02 06:00:00  29678.34  34200.00  29555.99  33769.52  2144.592516      60152.0  31941.412694
2  BTC/USD 2021-01-03 06:00:00  33769.52  34812.93  32300.61  32908.02  1838.695433      58725.0  33505.269474
3  BTC/USD 2021-01-04 06:00:00  32907.36  33496.03  27900.00  30441.57  2711.189503      69226.0  31267.965121
4  BTC/USD 2021-01-05 06:00:00  30461.84  35851.42  29927.55  35063.00  1756.751333      61880.0  33151.367357
    symbol           timestamp     open     high     low    close       volume  trade_count         vwap
0  ETH/USD 2021-01-01 06:00:00   742.15   745.00  716.65   733.65   6534.81013       6514.0   731.807780
1  ETH/USD 2021-01-02 06:00:00   735.29   791.08  725.44   786.88  16071.74933      15214.0   769.787732
2  ETH/USD 2021-01-

# Generate Trading Signals Using VWAP & SuperTrend 

Apply my VWAP & SuperTrend Trading Logic

In [73]:
def calculate_supertrend(data, period=5, multiplier=2.5):
    """Calculate Supertrend indicator"""
    data = data.copy()
    
    # Calculate ATR (Average True Range)
    high_low = data['high'] - data['low']
    high_close = np.abs(data['high'] - data['close'].shift())
    low_close = np.abs(data['low'] - data['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.rolling(period).mean()
    
    # Calculate basic upper and lower bands
    hl2 = (data['high'] + data['low']) / 2
    upper_band = hl2 + (multiplier * atr)
    lower_band = hl2 - (multiplier * atr)
    
    # Initialize Supertrend
    supertrend = pd.Series(np.zeros(len(data)), index=data.index)
    final_upper = pd.Series(np.zeros(len(data)), index=data.index)
    final_lower = pd.Series(np.zeros(len(data)), index=data.index)
    
    # Calculate Supertrend
    for i in range(period, len(data)):
        if data['close'].iloc[i-1] <= final_upper.iloc[i-1]:
            final_upper.iloc[i] = min(upper_band.iloc[i], final_upper.iloc[i-1])
        else:
            final_upper.iloc[i] = upper_band.iloc[i]
        if data['close'].iloc[i-1] >= final_lower.iloc[i-1]:
            final_lower.iloc[i] = max(lower_band.iloc[i], final_lower.iloc[i-1])
        else:
            final_lower.iloc[i] = lower_band.iloc[i]
        if (supertrend.iloc[i-1] > 0 and data['close'].iloc[i] > final_upper.iloc[i-1]) or \
           (supertrend.iloc[i-1] <= 0 and data['close'].iloc[i] > final_upper.iloc[i]):
            supertrend.iloc[i] = 1  # Uptrend
        elif (supertrend.iloc[i-1] < 0 and data['close'].iloc[i] < final_lower.iloc[i-1]) or \
             (supertrend.iloc[i-1] >= 0 and data['close'].iloc[i] < final_lower.iloc[i]):
            supertrend.iloc[i] = -1  # Downtrend
        else:
            supertrend.iloc[i] = supertrend.iloc[i-1]
    
    return supertrend


def generate_vwap_super_signals(data, supertrend_period=5, supertrend_multiplier=2.5):
    """Generate trading signals using VWAP and Supertrend confirmation"""
    data = data.copy()
    data = data.reset_index()
    
    # Create signals dataframe
    signals = pd.DataFrame(index=data.index)
    signals['symbol'] = data['symbol']
    signals['timestamp'] = data['timestamp']
    signals['close'] = data['close']
    signals['vwap'] = data['vwap']
    
    # Calculate Supertrend
    supertrend = calculate_supertrend(data, supertrend_period, supertrend_multiplier)
    signals['supertrend'] = supertrend


    # # Ensure alignment
    # supertrend = supertrend.reset_index(level=0, drop=True)
    # signals['supertrend'] = supertrend.iloc[:, 0]  # Extract first column
    
    # Simplified VWAP signals
    signals['raw_signal'] = 0
    signals.loc[signals['close'] > signals['vwap'], 'raw_signal'] = 1  # Buy
    signals.loc[signals['close'] < signals['vwap'], 'raw_signal'] = -1  # Sell
    
    # Apply Supertrend confirmation
    signals['signal'] = 0
    signals.loc[(signals['raw_signal'] == 1) & (signals['supertrend'] > 0), 'signal'] = 1
    signals.loc[(signals['raw_signal'] == -1) & (signals['supertrend'] < 0), 'signal'] = -1
    
    # Handle symbol boundaries
    signals['symbol_change'] = signals['symbol'] != signals['symbol'].shift(1)
    signals.loc[signals['symbol_change'], 'supertrend'] = np.nan
    signals.loc[signals['symbol_change'], 'signal'] = 0
    
    # Clean up
    signals.drop(columns=['symbol_change', 'raw_signal'], inplace=True)
    signals.set_index(['symbol', 'timestamp'], inplace=True)
    
    return signals



## Generate Signals

In [74]:
# Apply to BTC and ETH data
btc_signals = generate_vwap_super_signals(btc_data, supertrend_period=5, supertrend_multiplier=2.5)
eth_signals = generate_vwap_super_signals(eth_data, supertrend_period=5, supertrend_multiplier=2.5)

# Reset index to align them properly
btc_data = btc_data.reset_index(drop=True)
eth_data = eth_data.reset_index(drop=True)
btc_signals = btc_signals.reset_index(drop=True)
eth_signals = eth_signals.reset_index(drop=True)

# Join data
btc_data = btc_data.join(btc_signals[['signal', 'supertrend']])
eth_data = eth_data.join(eth_signals[['signal', 'supertrend']])

# # Display results
print("\nBTC Data Sample:")
print(btc_data.head())

print("\nETH Data Sample:")
print(eth_data.head())



BTC Data Sample:
    symbol           timestamp      open      high       low     close       volume  trade_count          vwap  signal  supertrend
0  BTC/USD 2021-01-01 06:00:00  29255.71  29682.29  28707.56  29676.79   848.874030      29639.0  29316.444625       0         NaN
1  BTC/USD 2021-01-02 06:00:00  29678.34  34200.00  29555.99  33769.52  2144.592516      60152.0  31941.412694       0         0.0
2  BTC/USD 2021-01-03 06:00:00  33769.52  34812.93  32300.61  32908.02  1838.695433      58725.0  33505.269474       0         0.0
3  BTC/USD 2021-01-04 06:00:00  32907.36  33496.03  27900.00  30441.57  2711.189503      69226.0  31267.965121       0         0.0
4  BTC/USD 2021-01-05 06:00:00  30461.84  35851.42  29927.55  35063.00  1756.751333      61880.0  33151.367357       0         0.0

ETH Data Sample:
    symbol           timestamp     open     high     low    close       volume  trade_count         vwap  signal  supertrend
0  ETH/USD 2021-01-01 06:00:00   742.15   745.00  716

# Feature Engineering for ML Model

Create input features using VWAP-SuperTrend Signal Strength, Risk Metrics, and Asset Correlations.

In [75]:
def feature_engineering(data):
    """
    Feature engineering for crypto trading strategy on a single asset.

    Parameters:
    - data (pd.DataFrame): DataFrame containing OHLCV and signal data for a single asset.

    Returns:
    - pd.DataFrame: Processed DataFrame with additional features.
    """
    # Convert timestamp to datetime if not already
    data['timestamp'] = pd.to_datetime(data['timestamp'])
    
    # Set timestamp as index
    data = data.set_index('timestamp')

    # Calculate Daily Returns
    data['returns'] = data['close'].pct_change()

    # Calculate Volatility (30-day rolling standard deviation)
    data['volatility'] = data['returns'].rolling(window=30).std()

    # Drawdown Calculation
    rolling_max = data['close'].cummax()
    data['drawdown'] = (data['close'] - rolling_max) / rolling_max
    data['max_drawdown'] = data['drawdown'].rolling(window=30, min_periods=1).min()

    # Select relevant columns
    features = data[['signal', 'returns', 'volatility', 'max_drawdown']].dropna()

    return features

# Apply feature engineering to BTC and ETH data
btc_features = feature_engineering(btc_data)
eth_features = feature_engineering(eth_data)

# Reset index so timestamp becomes a regular column
btc_features = btc_features.reset_index()
eth_features = eth_features.reset_index()

# Display first few rows
print(btc_features.head())
print(eth_features.head())


            timestamp  signal   returns  volatility  max_drawdown
0 2021-01-31 06:00:00       0  0.000761    0.062748     -0.221947
1 2021-02-01 06:00:00       0  0.006576    0.057614     -0.221947
2 2021-02-02 06:00:00       0  0.073531    0.058818     -0.221947
3 2021-02-03 06:00:00       1  0.031129    0.056998     -0.221947
4 2021-02-04 06:00:00       0 -0.014293    0.050286     -0.221947
            timestamp  signal   returns  volatility  max_drawdown
0 2021-01-31 06:00:00       0 -0.021776    0.097499     -0.215613
1 2021-02-01 06:00:00       0  0.051174    0.097210     -0.215613
2 2021-02-02 06:00:00       0  0.111833    0.072247     -0.215613
3 2021-02-03 06:00:00       1  0.073483    0.070358     -0.215613
4 2021-02-04 06:00:00       1 -0.014788    0.068249     -0.215613


# Train the Machine Learning Model

In [76]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

def train_random_forest(data, target):
    """
    Trains a Random Forest model and applies predictions to the dataset.

    Parameters:
    - data (pd.DataFrame): Input dataset with features and target.
    - target (str): Name of the target column.

    Returns:
    - model (RandomForestClassifier): Trained Random Forest model.
    - data (pd.DataFrame): DataFrame with added ML_Signal predictions.
    """

    # Automatically detect features (all numerical columns except target)
    features = [col for col in data.select_dtypes(include=['number']).columns if col != target]

    # Debugging: Print selected features
    print(f"Using features: {features} | Target: {target}")

    # Split dataset into train and test sets (80% train, 20% test)
    X_train, X_test, y_train, y_test = train_test_split(
        data[features], data[target], test_size=0.2, random_state=42
    )

    # Initialize and train the Random Forest model
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    # Apply predictions to the full dataset
    data["ML_Signal"] = model.predict(data[features])

    return model, data

# Train the model on btc_data and eth_data
btc_model, btc_data = train_random_forest(btc_data, "signal")
eth_model, eth_data = train_random_forest(eth_data, "signal")

# Display the first few rows
print(btc_data.head())
print(eth_data.head())


Using features: ['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'supertrend'] | Target: signal
Using features: ['open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'supertrend'] | Target: signal
    symbol           timestamp      open      high       low     close       volume  trade_count          vwap  signal  supertrend  ML_Signal
0  BTC/USD 2021-01-01 06:00:00  29255.71  29682.29  28707.56  29676.79   848.874030      29639.0  29316.444625       0         NaN          0
1  BTC/USD 2021-01-02 06:00:00  29678.34  34200.00  29555.99  33769.52  2144.592516      60152.0  31941.412694       0         0.0          0
2  BTC/USD 2021-01-03 06:00:00  33769.52  34812.93  32300.61  32908.02  1838.695433      58725.0  33505.269474       0         0.0          0
3  BTC/USD 2021-01-04 06:00:00  32907.36  33496.03  27900.00  30441.57  2711.189503      69226.0  31267.965121       0         0.0          0
4  BTC/USD 2021-01-05 06:00:00  30461.84  35851.42  29927.55  35

# Portfolio Optimization with pyportfolioopt

Use pyportfolioopt to generate target allocations.

In [77]:
from pypfopt.efficient_frontier import EfficientFrontier
from pypfopt.risk_models import risk_matrix
from pypfopt.expected_returns import mean_historical_return

# Combine BTC/USD and ETH/USD
prices = pd.concat([btc_data["close"], eth_data["close"]], axis=1)
prices.columns = ["BTC", "ETH"]

# Calculate expected returns and risk matrix
mu = mean_historical_return(prices)
S = risk_matrix(prices)

# Optimize portfolio allocation
ef = EfficientFrontier(mu, S)
weights = ef.max_sharpe()
cleaned_weights = ef.clean_weights()

print(cleaned_weights)  # Allocated portfolio weights


OrderedDict([('BTC', 1.0), ('ETH', 0.0)])


# Execute Live Paper Trades on Alpaca

Now, execute buy/sell orders based on the ML model and portfolio optimization.

In [80]:
def place_order(symbol, qty, side):
    from alpaca.trading.requests import MarketOrderRequest
    from alpaca.trading.enums import OrderSide, TimeInForce

    order = MarketOrderRequest(
        symbol=symbol,
        qty=qty,
        side=side,
        time_in_force=TimeInForce.GTC  # Good till canceled
    )

    try:
        response = trade_client.submit_order(order_data=order)
        print(f"Order placed: {symbol} - {side} {qty}")
    except Exception as e:
        print(f"Error placing order: {e}")

# Execute trades for BTC/USD & ETH/USD based on ML predictions
for symbol, data in zip(["BTC/USD", "ETH/USD"], [btc_data, eth_data]):
    last_signal = data["ML_Signal"].iloc[-1]  # Get latest prediction
    weight = cleaned_weights[symbol.split('/')[0]]  # Get allocated weight

    if last_signal == 1:
        place_order(symbol, qty=weight * 10, side=OrderSide.BUY)  # Adjust qty based on weight
    elif last_signal == -1:
        place_order(symbol, qty=weight * 10, side=OrderSide.SELL)  # Adjust qty based on weight


Error placing order: {"code":40310000,"message":"order cost basis must not exceed the notional limit of $200000"}
Error placing order: {"code":40010001,"message":"qty must be \u003e 0"}
